# CG4002: PyTorch 1D-CNN Trainer & HLS Export
---
**Target:** Xilinx Ultra96-V2 (Vitis HLS)  
**Framework:** PyTorch  
**Architecture:** 1D-CNN (Input: 6 axes x 60 samples)
- PyTorch Conv1d expects input as `(Batch, Channels, Length)` -> `(Batch, 6, 60)`.
- This notebook handles the transposition automatically.

---
**Notebook Structure:**
1. Import dependencies
2. Data processing & augmentation
3. Model definition
4. Training
5. Export weights for HLS

## 1. Import dependencies & configuration

In [ ]:
# Install required packages (uncomment if needed)
# !pip install pandas numpy scikit-learn

  Using cached sympy-1.13.1-py3-none-any.whl.metadata (12 kB)
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------- ----------- 6.8/9.7 MB 38.1 MB/s eta 0:00:01
   ---------------------------------------- 9.7/9.7 MB 35.6 MB/s eta 0:00:00
Using cached sympy-1.13.1-py3-none-any.whl (6.2 MB)
  Attempting uninstall: sympy
    Found existing installation: sympy 1.14.0
    Uninstalling sympy-1.14.0:
      Successfully uninstalled sympy-1.14.0


In [ ]:
# Install PyTorch (uncomment if needed)
# %pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://download.pytorch.org/whl/cu118
  Using cached https://download.pytorch.org/whl/cu118/torchvision-0.22.1%2Bcu118-cp312-cp312-win_amd64.whl.metadata (6.3 kB)
  Using cached https://download.pytorch.org/whl/cu118/torchaudio-2.7.1%2Bcu118-cp312-cp312-win_amd64.whl.metadata (6.8 kB)
  Using cached https://download.pytorch.org/whl/cu118/torch-2.7.1%2Bcu118-cp312-cp312-win_amd64.whl.metadata (27 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
Using cached https://download.pytorch.org/whl/cu118/torchvision-0.22.1%2Bcu118-cp312-cp312-win_amd64.whl (5.5 MB)
Using cached https://download.pytorch.org/whl/cu118/torch-2.7.1%2Bcu118-cp312-cp312-win_amd64.whl (2817.2 MB)
Using cached https://download.pytorch.org/whl/cu118/torchaudio-2.7.1%2Bcu118-cp312-cp312-win_amd64.whl (4.1 MB)
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
  Attempting uninstall: sympy
    Found existing installation: sympy 1.13.1
    Uninstalling sympy-1.13.1:
      Successfully 

In [11]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# CONFIGURATION
WINDOW_SIZE = 60
NUM_AXES = 6
NUM_CLASSES = 6
BATCH_SIZE = 16
EPOCHS = 40
LEARNING_RATE = 0.001

# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## 2. Data processing & augmentation

### 2.1 Read and parse raw gesture data from text files

In [13]:
import pandas as pd
import glob
import os

# Map filenames to Class IDs (0-5)
CLASS_MAP = {
    "Raise hand and hold.txt": 0,
    "Shaking fist (3 shakes).txt": 1,
    "Vertical chop.txt": 2,
    "Circular stir.txt": 3,
    "Horizontal Swing.txt": 4,
    "Punch (forward thrust).txt": 5,
}

def parse_and_convert():
    all_data_rows = []
    files = glob.glob("*.txt")
    print(f"Found {len(files)} files to process.")
    global_measurement_id = 0
    for filename in files:
        if filename not in CLASS_MAP:
            print(f"Skipping {filename} (not in CLASS_MAP)")
            continue
        label_id = CLASS_MAP[filename]
        label_name = filename.replace(".txt", "")
        print(f"Processing '{filename}' -> Class {label_id}")
        with open(filename, 'r') as f:
            lines = f.readlines()
        is_recording = False
        seq_counter = 0
        for line in lines:
            line = line.strip()
            if "YPR(deg)-X" in line:
                is_recording = True
                global_measurement_id += 1
                seq_counter = 0
                continue 
            if "Collecting data" in line or line == "" or any(k.replace(".txt","") in line for k in CLASS_MAP.keys()):
                is_recording = False
                continue
            if is_recording:
                try:
                    parts = [float(x.strip()) for x in line.split(',')]
                    if len(parts) == 6:
                        all_data_rows.append({
                            "measurement_id": global_measurement_id,
                            "sequence_id": seq_counter,
                            "label_id": label_id,
                            "label": label_name,
                            "gyro_x": parts[0],
                            "gyro_y": parts[1],
                            "gyro_z": parts[2],
                            "acc_x": parts[3],
                            "acc_y": parts[4],
                            "acc_z": parts[5]
                        })
                        seq_counter += 1
                except ValueError:
                    continue
    if all_data_rows:
        df = pd.DataFrame(all_data_rows)
        df.to_csv("imudata.csv", index=False)
        print(f"\n✅ SUCCESS! Processed {global_measurement_id} distinct recordings.")
        print(f"Total rows: {len(df)}")
        print(f"Saved to: imudata.csv")
        print("\nSample Data:")
        print(df.head())
    else:
        print("❌ No valid data found.")

# Run only if needed
# parse_and_convert()

### 2.2 Resample gesture data to fixed window size

In [14]:
from scipy import signal

TARGET_LEN = WINDOW_SIZE
OUTPUT_CSV = "resampled_imudata.csv"

all_recordings = []
global_id = 0
files = glob.glob("*.txt")
for filename in files:
    if filename not in CLASS_MAP:
        continue
    label_id = CLASS_MAP[filename]
    label_name = filename.replace(".txt", "")
    with open(filename, 'r') as f:
        lines = f.readlines()
    current_block = []
    is_recording = False
    for line in lines:
        line = line.strip()
        if "YPR(deg)-X" in line:
            if current_block:
                all_recordings.append((pd.DataFrame(current_block, columns=['gyro_x','gyro_y','gyro_z','acc_x','acc_y','acc_z']), label_id, label_name, global_id))
                global_id += 1
            current_block = []
            is_recording = True
            continue 
        if "Collecting data" in line or line == "" or any(k.replace(".txt","") in line for k in CLASS_MAP.keys()):
            if current_block:
                all_recordings.append((pd.DataFrame(current_block, columns=['gyro_x','gyro_y','gyro_z','acc_x','acc_y','acc_z']), label_id, label_name, global_id))
                global_id += 1
            current_block = []
            is_recording = False
            continue
        if is_recording:
            try:
                parts = [float(x.strip()) for x in line.split(',')]
                if len(parts) == 6:
                    current_block.append(parts)
            except ValueError:
                continue
    if current_block:
        all_recordings.append((pd.DataFrame(current_block, columns=['gyro_x','gyro_y','gyro_z','acc_x','acc_y','acc_z']), label_id, label_name, global_id))
        global_id += 1

resampled_rows = []
for df, label_id, label_name, m_id in all_recordings:
    raw_data = df.values
    seq_len = len(raw_data)
    if seq_len < 10: continue
    resampled_data = signal.resample(raw_data, TARGET_LEN)
    for i in range(TARGET_LEN):
        resampled_rows.append({
            "measurement_id": m_id,
            "sequence_id": i,
            "label_id": label_id,
            "label": label_name,
            "gyro_x": resampled_data[i, 0],
            "gyro_y": resampled_data[i, 1],
            "gyro_z": resampled_data[i, 2],
            "acc_x": resampled_data[i, 3],
            "acc_y": resampled_data[i, 4],
            "acc_z": resampled_data[i, 5]
        })
if resampled_rows:
    final_df = pd.DataFrame(resampled_rows)
    final_df.to_csv(OUTPUT_CSV, index=False)
    print(f"\n✅ SUCCESS! Saved to {OUTPUT_CSV}")
    print(f"Total Rows: {len(final_df)}")
    print(f"Unique Gestures: {final_df['measurement_id'].nunique()}")
else:
    print("❌ Error: No data found.")


✅ SUCCESS! Saved to resampled_imudata.csv
Total Rows: 1800
Unique Gestures: 30


### 2.3 Data augmentation (jitter, scaling, shift, time warping)

In [15]:
AUGMENT_FACTOR = 10
OUTPUT_CSV = "augmented_imudata.csv"
raw_recordings = []
files = glob.glob("*.txt")
for filename in files:
    if filename not in CLASS_MAP: continue
    label_id = CLASS_MAP[filename]
    label_name = filename.replace(".txt", "")
    with open(filename, 'r') as f:
        lines = f.readlines()
    current_block = []
    is_recording = False
    for line in lines:
        line = line.strip()
        if "YPR(deg)-X" in line:
            if current_block:
                raw_recordings.append((pd.DataFrame(current_block, columns=['gyro_x','gyro_y','gyro_z','acc_x','acc_y','acc_z']), label_id, label_name))
            current_block = []
            is_recording = True
            continue
        if "Collecting data" in line or line == "" or any(k.replace(".txt","") in line for k in CLASS_MAP.keys()):
            if current_block:
                raw_recordings.append((pd.DataFrame(current_block, columns=['gyro_x','gyro_y','gyro_z','acc_x','acc_y','acc_z']), label_id, label_name))
            current_block = []
            is_recording = False
            continue
        if is_recording:
            try:
                parts = [float(x.strip()) for x in line.split(',')]
                if len(parts) == 6: current_block.append(parts)
            except ValueError: continue
    if current_block:
        raw_recordings.append((pd.DataFrame(current_block, columns=['gyro_x','gyro_y','gyro_z','acc_x','acc_y','acc_z']), label_id, label_name))

final_rows = []
global_id = 0
for df, label_id, label_name in raw_recordings:
    raw_data = df.values
    seq_len = len(raw_data)
    if seq_len < 10: continue
    for i in range(AUGMENT_FACTOR):
        aug_data = raw_data.copy()
        if i > 0:
            noise = np.random.normal(0, 0.05, aug_data.shape)
            aug_data += noise
            scale = np.random.uniform(0.8, 1.2)
            aug_data *= scale
            shift = np.random.uniform(-0.1, 0.1, size=(1, 6))
            aug_data += shift
        if i > 5 and seq_len > 20:
            crop_start = np.random.randint(0, int(seq_len*0.1))
            crop_end = seq_len - np.random.randint(0, int(seq_len*0.1))
            aug_data = aug_data[crop_start:crop_end]
        resampled_data = signal.resample(aug_data, WINDOW_SIZE)
        for t in range(WINDOW_SIZE):
            final_rows.append({
                "measurement_id": global_id,
                "sequence_id": t,
                "label_id": label_id,
                "label": label_name,
                "gyro_x": resampled_data[t,0], "gyro_y": resampled_data[t,1], "gyro_z": resampled_data[t,2],
                "acc_x": resampled_data[t,3], "acc_y": resampled_data[t,4], "acc_z": resampled_data[t,5]
            })
        global_id += 1
df_final = pd.DataFrame(final_rows)
df_final.to_csv(OUTPUT_CSV, index=False)
print(f"\n✅ SUCCESS!")
print(f"Original Gestures: {len(raw_recordings)}")
print(f"Augmented Gestures: {global_id} (Target: ~{len(raw_recordings) * AUGMENT_FACTOR})")
print(f"Total Rows: {len(df_final)}")
print(f"Saved to: {OUTPUT_CSV}")


✅ SUCCESS!
Original Gestures: 30
Augmented Gestures: 300 (Target: ~300)
Total Rows: 18000
Saved to: augmented_imudata.csv


## 3. Model definition & training

In [16]:
# Load and preprocess augmented data for training
CSV_PATH = "augmented_imudata.csv"
print(f"Loading {CSV_PATH}...")
df = pd.read_csv(CSV_PATH)
feature_cols = ['gyro_x', 'gyro_y', 'gyro_z', 'acc_x', 'acc_y', 'acc_z']
X_flat = df[feature_cols].values
y_flat = df['label_id'].values
num_samples = len(df) // WINDOW_SIZE
print(f"Detected {num_samples} total gesture samples.")
X_all = X_flat.reshape(num_samples, WINDOW_SIZE, 6)
y_all = y_flat[::WINDOW_SIZE] 
N, L, C = X_all.shape
X_2d = X_all.reshape(-1, C)
scaler = StandardScaler()
X_scaled_2d = scaler.fit_transform(X_2d)
X_scaled = X_scaled_2d.reshape(N, L, C)
X_pytorch = np.transpose(X_scaled, (0, 2, 1))
X_train, X_test, y_train, y_test = train_test_split(
    X_pytorch, y_all, test_size=0.2, random_state=42, stratify=y_all
)
train_dataset = TensorDataset(torch.from_numpy(X_train).float().to(device), torch.from_numpy(y_train).long().to(device))
test_dataset = TensorDataset(torch.from_numpy(X_test).float().to(device), torch.from_numpy(y_test).long().to(device))
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
print(f"✅ Data Ready! Train shape: {X_train.shape} | Test shape: {X_test.shape}")

Loading augmented_imudata.csv...
Detected 300 total gesture samples.
✅ Data Ready! Train shape: (240, 6, 60) | Test shape: (60, 6, 60)


In [17]:
import torch.nn as nn

class GestureCNN(nn.Module):
    def __init__(self):
        super(GestureCNN, self).__init__()
        
        # Layer 1: Input 6 -> Output 16
        self.conv1 = nn.Conv1d(in_channels=6, out_channels=16, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool1d(kernel_size=2) # 60 -> 30
        
        # Layer 2: Input 16 -> Output 32
        self.conv2 = nn.Conv1d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool1d(kernel_size=2) # 30 -> 15
        
        # Flatten Calculation:
        # Final Length = 15
        # Final Channels = 32
        # Flatten Size = 15 * 32 = 480
        self.flatten_size = 15 * 32
        
        # Dense Layers
        self.fc1 = nn.Linear(self.flatten_size, 32)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(32, NUM_CLASSES) # Output 6 classes
        
    def forward(self, x):
        # Block 1
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)
        
        # Block 2
        x = self.conv2(x)
        x = self.relu2(x)
        x = self.pool2(x)
        
        # Classifier
        x = x.view(-1, self.flatten_size) # Flatten
        x = self.fc1(x)
        x = self.relu3(x)
        x = self.fc2(x)
        return x

# Instantiate and move to GPU
model = GestureCNN().to(device)
print(f"✅ Model Initialized on {device}")
print(model)

✅ Model Initialized on cuda
GestureCNN(
  (conv1): Conv1d(6, 16, kernel_size=(3,), stride=(1,), padding=(1,))
  (relu1): ReLU()
  (pool1): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv1d(16, 32, kernel_size=(3,), stride=(1,), padding=(1,))
  (relu2): ReLU()
  (pool2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=480, out_features=32, bias=True)
  (relu3): ReLU()
  (fc2): Linear(in_features=32, out_features=6, bias=True)
)


In [18]:
import torch.optim as optim

# Hyperparameters
EPOCHS = 40
LEARNING_RATE = 0.001

# Setup
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(f"🚀 Starting training for {EPOCHS} epochs...")

for epoch in range(EPOCHS):
    # --- TRAIN ---
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    train_acc = 100 * correct / total
    avg_loss = running_loss / len(train_loader)
    
    # --- VALIDATE ---
    model.eval()
    test_correct = 0
    test_total = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            test_total += labels.size(0)
            test_correct += (predicted == labels).sum().item()
            
    test_acc = 100 * test_correct / test_total
    
    # Log every 5 epochs
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | Train Acc: {train_acc:.1f}% | Test Acc: {test_acc:.1f}%")

print("✅ Training Complete.")

🚀 Starting training for 40 epochs...
Epoch 5/40 | Loss: 0.7146 | Train Acc: 97.9% | Test Acc: 96.7%
Epoch 10/40 | Loss: 0.0375 | Train Acc: 100.0% | Test Acc: 100.0%
Epoch 15/40 | Loss: 0.0098 | Train Acc: 100.0% | Test Acc: 100.0%
Epoch 20/40 | Loss: 0.0050 | Train Acc: 100.0% | Test Acc: 100.0%
Epoch 25/40 | Loss: 0.0030 | Train Acc: 100.0% | Test Acc: 100.0%
Epoch 30/40 | Loss: 0.0020 | Train Acc: 100.0% | Test Acc: 100.0%
Epoch 35/40 | Loss: 0.0015 | Train Acc: 100.0% | Test Acc: 100.0%
Epoch 40/40 | Loss: 0.0011 | Train Acc: 100.0% | Test Acc: 100.0%
✅ Training Complete.


## 4. Export to C++ Header (`weights.h`)
This extracts the trained parameters from `model.state_dict()` and formats them for your HLS code.

In [ ]:
def export_pytorch_weights(model, filename="gesture_cnn_weights.h"):
    print(f"Exporting weights to {filename}...")
    
    # Get CPU state dict
    params = model.to('cpu').state_dict()
    
    with open(filename, 'w') as f:
        f.write(f"#ifndef WEIGHTS_H\n#define WEIGHTS_H\n\n")
        f.write("#include \"typedefs.h\"\n\n")
        
        total_params = 0
        
        for name, tensor in params.items():
            # Clean name (e.g., "conv1.weight" -> "conv1_w")
            clean_name = name.replace(".", "_").replace("weight", "w").replace("bias", "b")
            
            # Flatten data row-major
            data = tensor.numpy().flatten()
            size = len(data)
            total_params += size
            
            print(f"Processing {name} -> {clean_name} ({size} elements)")
            
            # Write Array Definition
            f.write(f"// PyTorch Layer: {name} (Shape: {tuple(tensor.shape)})\n")
            f.write(f"static const data_t {clean_name}[{size}] = {{\n")
            
            for i, val in enumerate(data):
                f.write(f"{val:.6f}")
                if i < size - 1:
                    f.write(", ")
                if (i + 1) % 10 == 0: 
                    f.write("\n    ")
            f.write("\n};\n\n")
            
        f.write("#endif // WEIGHTS_H\n")
        
    print(f"\nDone! Total parameters: {total_params}")
    print("File saved as 'gesture_cnn_weights.h'. Upload this to Vitis HLS.")

# Run Export
export_pytorch_weights(model, "gesture_cnn_weights.h")

Exporting weights to gesture_cnn_weights.h...
Processing conv1.weight -> conv1_w (288 elements)
Processing conv1.bias -> conv1_b (16 elements)
Processing conv2.weight -> conv2_w (1536 elements)
Processing conv2.bias -> conv2_b (32 elements)
Processing fc1.weight -> fc1_w (15360 elements)
Processing fc1.bias -> fc1_b (32 elements)
Processing fc2.weight -> fc2_w (192 elements)
Processing fc2.bias -> fc2_b (6 elements)

Done! Total parameters: 17462
File saved as 'weights.h'. Upload this to Vitis HLS.
